# Threshold Method

This method finds a threshold value based on the climatological median for each pixel (or region) and then compares the daily data to the threshold. Values above the threshold are considered blooms and values below are not blooms. This step is testing different threshold values to determine what threshold most accurately models phytoplankton blooms on the NES. For testing, we chose 5-30% because these are values commonly used in bloom phenology literature. We then test these different threshold values on the daily data throughout the time series to determine which threshold is best for our region and for identifying the bloom metrics in question. 

This section includes functions to find the threshold values based on the climatology, create a mask to only include bloom values in an array, and function to clip data to smaller regions for different spatial comparisons to the mean. It culminates in a chosen threshold percentage for the entire NES region.

#### Import Python Libraries and datasets

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
from scipy import stats
from collections import defaultdict

In [2]:
#Daily chlorophyll data and regional zarr files
daily_data = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_D8.zarr')
MABN = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_D8.zarr')
GB = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_D8.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_D8.zarr')
GOME = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_D8.zarr')

In [18]:
#Region shapefiles
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
MAB_north_loc['geometry'] = MAB_north_loc.buffer(0)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

#### Function for determining the climatological threshold value

In [4]:
def threshold_value(thld=0.1, path=None):
    """
    Calculates the threshold value for chlorophyll-a based on a median baseline provided by the regional climatology.

    If no file path is provided, the path defaults to finding and reading the annual climatology file for the Northeast Shelf (NES) region. 
    The threshold is calculated by finding the percentage above the climatological CHL median for each pixel in the region.
    Must have access to NEFSC utilities functions to run this function without the path argument.

    Args:
        thld (float, optional): The fraction value of the percentage above the median. This value defaults to 0.1 (10%).
        path (str, optional): The path to netCDF file used to calculate the threshold value. Defaults to None.

    Returns:
        xarray.DataArray: A spatial array containing the threshold values for each coordinate based on the median CHL value
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return thld_value

In [5]:
def spatial_threshold_value(shapefile_geometry=None,thld=0.1,path=None,regions=None,region_col='Region',ordered_region_names=None,default_shapefile_path='https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip'):
    """
    Creates a threshold value for a spatially averaged area.

    Using the threshold value, this function creates a spatially averaged threshold value for a region for use in other analysis.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, optional): Shapefile of the region in question. Defaults to None
        thld (float, optional): Percentage for threshold calcultion. Defaults to 0.1.
        path (str, optional): Path to climatology file. Defaults to None.
        regions (list or str, optional): A subset of region names to filter by. Defaults to None
        region_col (str, optional): The column name of the shapefile containing region names.
        ordered_region_names (list, optional): The list of names of the region in the shapefile if not already a column in the shapefile. Defaults to None
        default_shapefile_path (str, optional): Path to the default shapefile
    
    Returns:
        float. Value of the regionally averaged chlorophyll threshold.
    """
    # STEP 1: Load the shapefile geometry
    if shapefile_geometry is None:
        shapefile = gpd.read_file(default_shapefile_path)
        ordered_regions = ['Middle Atlantic Bight South','Middle Atlantic Bight North', 'Georges Bank', 'Gulf of Maine West', 'Gulf of Maine East']
        shapefile['Region'] = ordered_regions
    else:
        shapefile = shapefile_geometry.copy()
    if shapefile.crs is None:
        shapefile = shapefile.set_crs("EPSG:4326")
    else:
        shapefile = shapefile.to_crs("EPSG:4326")
    if ordered_region_names is not None:
        if len(ordered_region_names) != len(shapefile):
            raise ValueError("The list of names provided does not match the number of rows in the shapefile")
        shapefile['Region'] = ordered_region_names
        region_col = 'Region'

    # STEP 2: Subset the shapefile
    if regions is not None:
        if isinstance(regions,str):
            regions = [regions]
        shapefile = shapefile[shapefile[region_col].isin(regions)]
        if shapefile.empty:
            raise ValueError(f"None of the provided regions {regions} were found in the shapefile")
    
    # STEP 3: Load and prepare threshold data
    threshold = threshold_value(thld=thld,path=path)
    threshold.rio.write_crs("EPSG:4326",inplace=True)
    threshold.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    
    # STEP 4: Build the Dataset
    results = []
    for _, row in shapefile.iterrows():
        region_name = row[region_col]
        region_geometry = [mapping(row.geometry)]
        try:
            clipped_thld = threshold.rio.clip(region_geometry, shapefile.crs, drop=True)
            clipped_thld = clipped_thld.mean(dim=['lat','lon']).item()
            results.append({'Region':region_name, 'Threshold': clipped_thld})
        except Exception as e:
            print(f"Skipping {region_name} due to processing error")
            results.append({'Region': region_name, 'Threshold': None})

    return pd.DataFrame(results)

In [6]:
#Calculating threshold and median values for each region
threshold_10 = spatial_threshold_value(path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
median_climatology = spatial_threshold_value(thld=0, path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')

MABS_thld = threshold_10['Threshold'][0]
MABN_thld = threshold_10['Threshold'][1]
GB_thld = threshold_10['Threshold'][2]
GOMW_thld = threshold_10['Threshold'][3]
GOME_thld = threshold_10['Threshold'][4]
MABS_median = median_climatology['Threshold'][0]
MABN_median = median_climatology['Threshold'][1]
GB_median = median_climatology['Threshold'][2]
GOMW_median = median_climatology['Threshold'][3]
GOME_median = median_climatology['Threshold'][4]

In [7]:
#Calculating the threshold and median values for each pixel in the subset.
subset_thld = threshold_value(path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\subset_climatology.nc')
subset_thld = subset_thld.squeeze()
subset_daily = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\subset_daily_data.zarr')
subset_daily = subset_daily.chunk({'time': -1})
subset_med = threshold_value(thld=0,path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\subset_climatology.nc')
subset_med = subset_med.squeeze()

#### Plotting the Climatological Threshold
To begin visualizing the data, we plot the climatological median threshold value on a map to see the spatial differences across the region. Notably, the Georges Bank area and estuaries typically have a higher median threshold value, while the open ocean has a relatively low threshold value. This means that blooms are more likely to be detected in open ocean regions since Georges Bank and estuaries have naturally higher primary productivity.

Maps produced also can have the regions boxed and labelled if "boxes" and "box_labels" are True, respectively.

![Chlorophyll a Climatological Median map with region boundaries labelled](Figures/median_and_study_locations.png)

In [ ]:
#Map with chlorophyll median

boxes = True
box_labels = True

shapefile_geometry = [MAB_south_loc, MAB_north_loc, GB_whole_loc, GOM_west_loc, GOM_east_loc]
shapefile_names = ['MAB South', 'MAB North', 'Georges Bank', 'GOM West', 'GOM East']
region_titles = ['Middle Atlantic Bight South', 'Middle Atlantic Bight North', 'Georges Bank', 'Gulf of Maine West', 'Gulf of Maine East']
colors = ['gold', 'cyan', 'darkorange', 'mediumorchid', 'dodgerblue']
clim_med = threshold_value(path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc', thld=0).squeeze()
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
fig = plt.figure(figsize=(20,8))
map_projection = cartopy.crs.PlateCarree()
ax = plt.axes(projection=map_projection)
ax.set_extent([-77,-65,35,45])
im = plt.pcolormesh(clim_med.lon,
                    clim_med.lat,
                    clim_med,
                    cmap=cmocean.cm.algae,
                    norm=LogNorm(vmin=0.1, vmax=10.0),
                    transform=cartopy.crs.PlateCarree(),
                    zorder=1
)
custom_ticks = [0.1,0.3,1,3,10]
cb = plt.colorbar(im,shrink=0.8,ticks=custom_ticks,format='%g',pad=0.01) #$ $ makes it a LaTEX function so it actually formats as an equation
cb.set_label(label='Chlorophyll a Concentration ($mg/m^3$)',fontsize=12)
if boxes is True:
    black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
    white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
    MAB_south_loc.boundary.plot(ax=ax, color='gold', linewidth=3, path_effects=black_halo, zorder=2, label=region_titles[0])
    MAB_north_loc.boundary.plot(ax=ax, color='cyan', linewidth=3, path_effects=black_halo, zorder=2, label=region_titles[1])
    GB_whole_loc.boundary.plot(ax=ax, color='darkorange', linewidth=3, path_effects=black_halo, zorder=2, label=region_titles[2])
    GOM_west_loc.boundary.plot(ax=ax, color='mediumorchid', linewidth=3, path_effects=black_halo, zorder=2, label=region_titles[3])
    GOM_east_loc.boundary.plot(ax=ax, color='dodgerblue', linewidth=3, path_effects=black_halo, zorder=2, label=region_titles[4])
    ax.legend(fontsize=16,loc='lower right')

if box_labels is True:
    for gdf,title,color in zip(shapefile_geometry,shapefile_names,colors):
        for idx, row in gdf.iterrows():
            rep_point = row.geometry.representative_point()
            if title == 'MAB South':
                rot_angle = 60
                x_offset, y_offset = (16,-3)
            elif title == "Georges Bank" or title == "GOM East":
                rot_angle=0
                x_offset, y_offset = (4,-2)
            else:
                rot_angle=0
                x_offset, y_offset = (0,0)
            ax.annotate(text=title,
                        xy=(rep_point.x, rep_point.y),
                        xytext=(x_offset,y_offset),
                        textcoords='offset points',
                        horizontalalignment='center',
                        fontsize=14,
                        color = 'black',
                        zorder=3,
                        fontweight='bold',
                        rotation=rot_angle,
                        bbox = dict(
                            boxstyle='round,pad=0.2',
                            facecolor='white',
                            alpha=0.5,
                            linewidth=1
                        )
            )
ax.add_feature(cartopy.feature.COASTLINE, linewidth=1,zorder=3)
ax.add_feature(cartopy.feature.LAND, zorder=3, facecolor='darkgrey')
ax.add_feature(bathym, facecolor='none', edgecolor='black', zorder=2) #Adding the shelf break line
states_provinces = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='50m',
    facecolor='none',
    edgecolor='gray',
    zorder = 3
)
ax.set_axisbelow(False)
ax.add_feature(states_provinces, linewidth=0.8)
gl = ax.gridlines(
    crs=cartopy.crs.PlateCarree(),
    draw_labels=True,
    color='dimgrey',
    xlocs = ticker.MultipleLocator(1.5),
    ylocs=ticker.MultipleLocator(1),
    zorder=10)
gl.top_labels = False
gl.right_labels = False
if hasattr(gl, 'geo_labels'):
    gl.geo_labels = False
ax.set_title('Chlorophyll a Climatological Median', fontsize=20)
plt.tight_layout()

#### Create a mask to filter data for bloom conditions
These functions create a mask to show only pixels that exceed the threshold values set by the threshold_value() function above. When bloom_mask_numeric is plotted, a daily map will show regions under the threshold value as white (no values) and the rest of the values as their raw value.

In [24]:
def bloom_mask(path=None,thld=0.1,clim_path=None,Boolean=False): #Produces True and False values
    """
    Creates a mask on the chlorophyll-a data to only include values above the threshold set by the climatological median.

    If no file path is provided, the function searches for all D8 files (8 day rolling mean) for the Northeast Shelf. 
    If a file path is provided, it is currently set to open zarr files. The code exists to open netCDFs as well, it just needs to be uncommented. 
    If no climatology file path is provided, the function searches for the annual climatology file of the Northeast Shelf region.
    The median chlorophyll-a values are then extracted and compared to the threshold value found from the regional climatology.
    If the median chlorophyll-a values exceed the threshold, the value is stored as true. Otherwise, it is stored as false.

    Args:
        path (str, optional): Path to the daily data (or other temporal resolution data). Defaults to None
        thld (float, optional): Fractional value for percentage to calculate the climatological threshold per coordinate. Defaults to 0.1 (10%)
        clim_path (str, optional): Path to regional climatology file. Defaults to None
        Boolean (bool, optional): Determines how values are stored. False keeps the actual values and marks false values as 0. True stores an array of True and False values. Defaults to False

    Returns:
        xarray.DataArray: A spatial array with Boolean values or floats for each coordinate based on the threshold value.
    """
    if path is None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NES
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path)
    if Boolean is True:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    else:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        clim_med_new = clim_med.isel(time=0, drop=True) #Removes time dimension from climatological mean
        is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

In [28]:
# Calculates the bloom mask for threshold values between 5% and 30% above the climatological median for daily (not spatially averaged) data.
climatology_path = r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc'
daily_path = r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_combined.zarr'

bloom_5 = bloom_mask(path=daily_path, thld=0.05, clim_path=climatology_path, Boolean=False)
bloom_10 = bloom_mask(path=daily_path, thld=0.1, clim_path=climatology_path, Boolean=False)
bloom_15 = bloom_mask(path=daily_path, thld=0.15, clim_path=climatology_path, Boolean=False)
bloom_20 = bloom_mask(path=daily_path, thld=0.2, clim_path=climatology_path, Boolean=False)
bloom_25 = bloom_mask(path=daily_path, thld=0.25, clim_path=climatology_path, Boolean=False)
bloom_30 = bloom_mask(path=daily_path, thld=0.3, clim_path=climatology_path, Boolean=False)


#### Histograms of Data

This step was done to investigate a randomly chosen 1 degree box and see how much of the chlorophyll-a data laid above the median threshold in question. Once the histograms were plotted for the actual chlorophyll-a values, lines representing that area's climatological median and calculated threshold values were overlaid to see how much data fell on either side of the thresholds. This was a higher spatial resolution method to begin testing the different threshold methods.

In [ ]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the data to a specified box for more precise spatial analysis.

    If no file path is provided, the function opens all D8 files for the Northeast Shelf. Currently it opens the D8_combined zarr file but can be changed to open the netCDF files.
    This function takes the daily data and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the daily data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median values for the spatial averaged area for the full time series of the data.
    """
    if path is None:
        #daily_data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        daily_data = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    daily_data_local = daily_data.CHL_median.sel(#Clips the CHL_median data to the specified spatial bounds
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    daily_data_local = daily_data_local.mean(dim=['lat','lon']) #Averages the data over the spatial bounds
    return daily_data_local

In [ ]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the climatology data to a specified box for more precise spatial analysis.

    If no file path is provided, the function searches for the annual climatology file for the Northeast Shelf.
    This function takes the median chlorophyll-a of the climatology and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the climatology data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median value for the spatial averaged area for the climatology.
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL')
        clim = xr.open_dataset(file[0])
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded


In [ ]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    """
    Creates a polygon shape for mapping

    This function takes in boundary coordinates and makes a shape to be used for plotting. 

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default

    Returns: 
        POLYGON
    """
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

#### Determining the percentage of datapoints that lie above each threshold
This was done to determine numerically the percentage of data that would be exceeding the threshold for each square of data. This helped to quantify what we could see on the previous histograms. It helped us eliminate 20%, 25%, and 30% from the list of viable thresholds. These thresholds excluded a large majority of the data in some regions and we wanted one threshold percentage for the whole NES region.

In [ ]:
def percent_above_thld(data,clipped_thld):
    """
    Calculates the percentage of data points that lie above a threshold value.
    
    The data provided must be clipped to a region prior to inputting into function. Otherwise the function will run it for the entire spatial data in the dataset.

    Args: 
        data (xarray.Dataset, required): Dataset for analysis. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for region

    Returns:
        float. The percentage of datapoints that lie above the threshold.
    """
    dataset = data['CHL_median']
    threshold = clipped_thld
    total_valid = dataset.notnull().sum()
    total_above = (dataset>threshold).sum()
    percent = (total_above/total_valid)*100
    return float(percent.compute())

In [ ]:
percent_5 = spatial_threshold_value(thld=0.05)
percent_15 = spatial_threshold_value(thld=0.15)
percent_20 = spatial_threshold_value(thld=0.2)
percent_25 = spatial_threshold_value(thld=0.25)
percent_30 = spatial_threshold_value(thld=0.3)

In [ ]:
MABS_5thld = percent_5['Threshold'][0]
MABN_5thld = percent_5['Threshold'][1]
GB_5thld = percent_5['Threshold'][2]
GOMW_5thld = percent_5['Threshold'][3]
GOME_5thld = percent_5['Threshold'][4]

MABS_15thld = percent_15['Threshold'][0]
MABN_15thld = percent_15['Threshold'][1]
GB_15thld = percent_15['Threshold'][2]
GOMW_15thld = percent_15['Threshold'][3]
GOME_15thld = percent_15['Threshold'][4]

MABS_20thld = percent_20['Threshold'][0]
MABN_20thld = percent_20['Threshold'][1]
GB_20thld = percent_20['Threshold'][2]
GOMW_20thld = percent_20['Threshold'][3]
GOME_20thld = percent_20['Threshold'][4]

MABS_25thld = percent_25['Threshold'][0]
MABN_25thld = percent_25['Threshold'][1]
GB_25thld = percent_25['Threshold'][2]
GOMW_25thld = percent_25['Threshold'][3]
GOME_25thld = percent_25['Threshold'][4]

MABS_30thld = percent_30['Threshold'][0]
MABN_30thld = percent_30['Threshold'][1]
GB_30thld = percent_30['Threshold'][2]
GOMW_30thld = percent_30['Threshold'][3]
GOME_30thld = percent_30['Threshold'][4]

In [ ]:
threshold_percentage = ['5%','10%','15%','20%','25%','30%']
MABS_thresholds = [MABS_5thld,MABS_thld,MABS_15thld,MABS_20thld,MABS_25thld,MABS_30thld]
MABN_thresholds = [MABN_5thld,MABN_thld,MABN_15thld,MABN_20thld,MABN_25thld,MABN_30thld]
GB_thresholds = [GB_5thld,GB_thld,GB_15thld,GB_20thld,GB_25thld,GB_30thld]
GOMW_thresholds = [GOMW_5thld,GOMW_thld,GOMW_15thld,GOMW_20thld,GOMW_25thld,GOMW_30thld]
GOME_thresholds = [GOME_5thld,GOME_thld,GOME_15thld,GOME_20thld,GOME_25thld,GOME_30thld]
MABS_loaded = MABS.load()
MABN_loaded = MABN.load()
GB_loaded = GB.load()
GOMW_loaded = GOMW.load()
GOME_loaded = GOME.load()
print("Finished loading data")
MABS_percent_above = []
MABN_percent_above = []
GB_percent_above = []
GOMW_percent_above = []
GOME_percent_above = []
for x in range(len(threshold_percentage)):
    MABS_above = percent_above_thld(MABS_loaded,MABS_thresholds[x])
    MABS_percent_above.append(MABS_above)
    MABN_above = percent_above_thld(MABN_loaded,MABN_thresholds[x])
    MABN_percent_above.append(MABN_above)
    GB_above = percent_above_thld(GB_loaded,GB_thresholds[x])
    GB_percent_above.append(GB_above)
    GOMW_above = percent_above_thld(GOMW_loaded,GOMW_thresholds[x])
    GOMW_percent_above.append(GOMW_above)
    GOME_above = percent_above_thld(GOME_loaded,GOME_thresholds[x])
    GOME_percent_above.append(GOME_above)
    print('Finished percentage')

data = {
    'Threshold_percentage': threshold_percentage,
    'MAB_South_Above_thld': MABS_percent_above,
    'MAB_North_Above_thld': MABN_percent_above,
    'Georges_Bank_Above_thld': GB_percent_above,
    'GOM_West_Above_thld': GOMW_percent_above,
    'GOM_East_Above_thld': GOME_percent_above
}
dataset = pd.DataFrame(data)
dataset.to_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Threshold_Diagnostic_Calculations.csv', mode='w', index=False)

#### Shapefile Analysis

We repeated the steps for the small 1-degree box analysis, this time spatially averaging the entire regions. In this step, the Gulf of Maine and Middle Atlantic Bight were separated into two regions each. This follows the shapefiles available but also covers spatial differences within the regions. From this analysis, we concluded that 5% was too low. To finaliaze the decision between 10% and 15%, we plotted the threshold value on a time series to observe what qualitatively made sense. We chose 10% because we felt that it encompassed enough of the bloom data without excluding smaller peaks. You could however choose 15% as well, both seemed to do well at representing the general trends of the NES.

![Histograms for each region of chlorophyll concentrations with thresholds overlaid](Figures\Threshold_Diagnostic_Graphs\all_regional_histograms_raw.png)

In [ ]:
#Clip the climatology data to each regional shapefile
clim_regional = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
clim_regional.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
clim_regional.rio.write_crs("epsg:4326", inplace=True)
clipped_MAB_s = clim_regional.rio.clip(MAB_south_loc.geometry, shapefile.crs, drop=True)
clim_MAB_s = clipped_MAB_s.CHL_median.mean(dim=['lat','lon'])
clipped_MAB_n = clim_regional.rio.clip(MAB_north_loc.geometry, shapefile.crs, drop=True)
clim_MAB_n = clipped_MAB_n.CHL_median.mean(dim=['lat','lon'])
clipped_GB = clim_regional.rio.clip(GB_whole_loc.geometry, shapefile.crs, drop=True)
clim_GB = clipped_GB.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_w = clim_regional.rio.clip(GOM_west_loc.geometry, shapefile.crs, drop=True)
clim_GOM_w = clipped_GOM_w.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_e = clim_regional.rio.clip(GOM_east_loc.geometry, shapefile.crs, drop=True)
clim_GOM_e = clipped_GOM_e.CHL_median.mean(dim=['lat','lon'])
clim_NES = clim_regional.rio.clip(NES.geometry, shapefile.crs, drop=True)
clim_NES = clim_NES.CHL_median.mean(dim=['lat','lon'])

In [ ]:
#Clipping the daily data to each regional shapefile
daily_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
daily_data.rio.write_crs("epsg:4326", inplace=True)
clipped_daily_MABS = daily_data.rio.clip(MAB_south_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_south = clipped_daily_MABS.CHL_median.mean(dim=['lat','lon'])
clipped_daily_MABN = daily_data.rio.clip(MAB_north_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_north = clipped_daily_MABN.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GB = daily_data.rio.clip(GB_whole_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GB_whole = clipped_daily_GB.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOMW = daily_data.rio.clip(GOM_west_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_west = clipped_daily_GOMW.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOME = daily_data.rio.clip(GOM_east_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_east = clipped_daily_GOME.CHL_median.mean(dim=['lat','lon'])
clipped_daily_NES = daily_data.rio.clip(NES.geometry.apply(mapping), shapefile.crs, drop=True)
NES_full = clipped_daily_NES.CHL_median.mean(dim=['lat','lon'])

In [ ]:
#Calculating the threshold values for each region based on the climatological median
local_chl = [MAB_south,MAB_north,GB_whole,GOM_west,GOM_east,NES_full]
clim_med = [clim_MAB_s,clim_MAB_n,clim_GB,clim_GOM_w,clim_GOM_e,clim_NES]
clim_5 = [clim_MAB_s*1.05,clim_MAB_n*1.05,clim_GB*1.05,clim_GOM_w*1.05,clim_GOM_e*1.05,clim_NES*1.05]
clim_10 = [clim_MAB_s*1.10,clim_MAB_n*1.10,clim_GB*1.10,clim_GOM_w*1.10,clim_GOM_e*1.10,clim_NES*1.10]
clim_15 = [clim_MAB_s*1.15,clim_MAB_n*1.15,clim_GB*1.15,clim_GOM_w*1.15,clim_GOM_e*1.15,clim_NES*1.15]
clim_20 = [clim_MAB_s*1.2,clim_MAB_n*1.2,clim_GB*1.2,clim_GOM_w*1.2,clim_GOM_e*1.2,clim_NES*1.2]
clim_25 = [clim_MAB_s*1.25,clim_MAB_n*1.25,clim_GB*1.25,clim_GOM_w*1.25,clim_GOM_e*1.25,clim_NES*1.25]
clim_30 = [clim_MAB_s*1.3,clim_MAB_n*1.3,clim_GB*1.3,clim_GOM_w*1.3,clim_GOM_e*1.3,clim_NES*1.3]

In [ ]:
fig, axes=plt.subplots(nrows=2,ncols=3,figsize=(16,5),sharey=True)

#Histograms
hist_axes = axes.flatten()
region_title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank",'Gulf of Maine West',"Gulf of Maine East"]
for i in range(len(region_title)):
    data = local_chl[i+1]
    ax = hist_axes[i]
    numpy_array = data.compute().values
    clean_data = numpy_array.ravel()
    clean_data = clean_data[~np.isnan(clean_data)]
    ax.hist(clean_data,bins=100, range=(0,4))
    ax.axvline(clim_med[i][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i][0], color='mediumorchid', label='5% Threshold')
    ax.axvline(clim_10[i][0], color='gold',label='10% Threshold')
    ax.axvline(clim_15[i][0], color='darkorange',label='15% Threshold')
    ax.axvline(clim_20[i][0], color='midnightblue',label='20% Threshold')
    ax.axvline(clim_25[i][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i][0], color='turquoise',label='30% Threshold')
    ax.set_xlabel("Chlorophyll a Concentration ($mg/m^3$)")
    ax.set_title(region_title[i])
handles, labels = hist_axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='lower center',
    ncol=3,
    fontsize=9,
    frameon=True,
    bbox_to_anchor=(0.5,-0.05),
)
plt.ylabel("Number of Data Points")
plt.tight_layout(rect=[0,0.07,1,0.95])
fig.suptitle("Chlorophyll a Concentration Histograms",fontsize=20,y=1.05)

#### Plotting the histograms centered on the median

Fix this function

In [ ]:
def percent_deviation(dataset,clipped_median,clipped_thld):
    """
    Calculates the amount of data that is a certain percentage deviated from the median.

    Args:
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        clipped_median (float, required): The pre-calculated median for the region. No defaults
        clipped_thld (float, required): The pre-calculated threshold for the region. No defaults
    
    Returns:
        numpy.ndarray. 
    """
    top = dataset.squeeze()-clipped_median
    fraction = top/clipped_thld
    percent_dev = fraction*100
    return percent_dev

Plotting the histograms

![Percent Deviation Histograms](Figures\Threshold_Diagnostic_Graphs\threshold_histograms.png)

In [ ]:
custom_bins=[0,5,10,15,20,25,30,35]
fig,axes=plt.subplots(2,3,figsize=(10,7),sharey=True)
axes=axes.flatten() #Creates a 1-D numpy array of indices for axes instead of a 2 by 3 array
data = [MABS_per_dev,MABN_per_dev,GB_per_dev,GOMW_per_dev,GOME_per_dev]
title = ['Middle Atlantic Bight South', 'Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
colors = ["gold",'cyan','darkorange','mediumorchid','dodgerblue']
for x in range(5):
    ax = axes[x]
    ax.hist(data[x], bins=custom_bins, facecolor = colors[x], edgecolor='black', linewidth=1)
    ax.set_xlabel("Percent Threshold")
    ax.tick_params(labelleft=True)
    ax.set_title(title[x])
axes[5].set_visible(False)
axes[0].set_ylabel("Amount of Data")
axes[3].set_ylabel("Amount of Data")
fig.suptitle("Percentage Threshold from Regional Median", fontsize=20)
plt.tight_layout()

From all of the above diagnostics, we established a 10% threshold above the median for all regions on the NES